# Large Point Clouds → Potree (EPT) + Local Server + Launcher URL
**Beginner-friendly, large-data oriented**  
**Last updated:** 2025-11-09

This notebook shows how to:
1) Convert big point clouds to **web-streamable formats** (EPT or classic Potree)  
2) Serve them from a **local CORS-enabled server** (localhost)  
3) Open them in the Potree launcher via a **URL like**:

`https://ida-mdc.gitlab.io/potree-launcher/?dataUrl=https://…/ept.json`

---
### TL;DR recommendations
- Prefer **EPT (Entwine Point Tiles)** for very large clouds; loads progressively on the web.
- If you already use **PotreeConverter**, you can still host `cloud.js` outputs.
- Ensure your server sends **CORS headers** when loading data from a different origin (like a remote GitLab page).

## 0) Prerequisites & installs
You can do the conversion with either **Conda** or **Docker**:

**Conda (recommended if you already use Conda/Mamba):**
```bash
mamba create -n ept -c conda-forge entwine pdal laz-perf laszip
mamba activate ept
```
*(On some platforms `entwine` may not be available; if so, use Docker below.)*

**Docker (portable, no local installs):**
```bash
# Entwine (EPT)
docker pull connormanning/entwine
# PotreeConverter (classic Potree)
docker pull potree/potreeconverter
```
**Input formats**: LAZ/LAS are ideal. If you have PLY/XYZ/CSV, convert with **PDAL** first.


## 1) Convert PLY/XYZ → LAZ (PDAL)
If your data isn’t LAZ/LAS, convert it using **PDAL**. Here are two common pipelines.

In [ ]:
# Write two PDAL pipeline JSON files for convenience (edit paths before running in a terminal)
ply_to_laz = r'''{
  "pipeline": [
    {"type":"readers.ply", "filename":"/path/to/input.ply"},
    {"type":"filters.reprojection", "out_srs":"EPSG:4978"},
    {"type":"writers.las", "filename":"/path/to/output.laz", "compression":"laszip"}
  ]
}'''

xyz_to_laz = r'''{
  "pipeline": [
    {"type":"readers.text", "filename":"/path/to/input.xyz", "header":"X Y Z"},
    {"type":"writers.las", "filename":"/path/to/output.laz", "compression":"laszip"}
  ]
}'''

with open('/mnt/data/pipeline_ply_to_laz.json','w',encoding='utf-8') as f:
    f.write(ply_to_laz)
with open('/mnt/data/pipeline_xyz_to_laz.json','w',encoding='utf-8') as f:
    f.write(xyz_to_laz)
print('Wrote: /mnt/data/pipeline_ply_to_laz.json, /mnt/data/pipeline_xyz_to_laz.json')


Run in your shell:
```bash
pdal pipeline /mnt/data/pipeline_ply_to_laz.json
# or
pdal pipeline /mnt/data/pipeline_xyz_to_laz.json
```
Now you have **output.laz** ready for web-tiling.

## 2) Build **EPT** with Entwine (recommended for large clouds)
### Option A — Conda
```bash
entwine build -i /path/to/output.laz -o /path/to/out_ept --threads 8
```
### Option B — Docker
```bash
docker run --rm -it -v /local/data:/data connormanning/entwine \
  build -i /data/output.laz -o /data/out_ept --threads 8
```
This creates a folder like `/path/to/out_ept/` containing `ept.json` and tiles.

## 3) (Alternative) Build classic Potree format with PotreeConverter
Use this if you specifically need Potree’s `cloud.js` structure.

**Docker:**
```bash
docker run --rm -it -v /local/data:/data potree/potreeconverter \
  /data/output.laz -o /data/out_potree --generate-page mycloud --title "My Cloud" --material ELEVATION
```
This produces `/path/to/out_potree/` with `cloud.js`.

## 4) Serve locally with **CORS** enabled
When loading data from a **remote launcher page** (different origin), browsers require **CORS** headers from your local server.

### Option A — Python tiny CORS server (simple)
Run this script in a terminal from the parent directory of your EPT/Potree folder:
```bash
python cors_server.py
```
It serves at `http://127.0.0.1:8000` and adds `Access-Control-Allow-Origin: *`.

In [ ]:
# Write a tiny CORS-enabled static server to /mnt/data/cors_server.py
cors_server = r'''from http.server import SimpleHTTPRequestHandler, HTTPServer
class CORSRequestHandler(SimpleHTTPRequestHandler):
    def end_headers(self):
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Methods', 'GET, OPTIONS')
        self.send_header('Access-Control-Allow-Headers', 'Range, Content-Type')
        return super().end_headers()

if __name__ == '__main__':
    import argparse, os
    ap = argparse.ArgumentParser()
    ap.add_argument('--port', type=int, default=8000)
    ap.add_argument('--directory', type=str, default='.')
    args = ap.parse_args()
    os.chdir(args.directory)
    httpd = HTTPServer(('0.0.0.0', args.port), CORSRequestHandler)
    print(f'Serving {os.getcwd()} at http://127.0.0.1:{args.port} (CORS enabled)')
    httpd.serve_forever()
'''
with open('/mnt/data/cors_server.py','w',encoding='utf-8') as f:
    f.write(cors_server)
print('Saved: /mnt/data/cors_server.py')


**Launch examples** (from a shell):
```bash
# Serve the parent directory that contains 'out_ept' (or 'out_potree')
python /mnt/data/cors_server.py --directory /path/to --port 8000
```
Then your data URLs will look like:
- **EPT**: `http://127.0.0.1:8000/out_ept/ept.json`
- **Potree (cloud.js)**: `http://127.0.0.1:8000/out_potree/cloud.js`

## 5) Create the Potree launcher URL
Use this URL shape (replace `dataUrl=` with your actual JSON):

`https://ida-mdc.gitlab.io/potree-launcher/?dataUrl=YOUR_ENCODED_URL`

Below, we’ll generate a fully encoded URL to paste into your browser.

In [ ]:
from urllib.parse import quote

launcher = 'https://ida-mdc.gitlab.io/potree-launcher/?dataUrl='
local_ept = 'http://127.0.0.1:8000/out_ept/ept.json'  # edit if needed
local_cloudjs = 'http://127.0.0.1:8000/out_potree/cloud.js'  # if using PotreeConverter

print('EPT launcher URL:')
print(launcher + quote(local_ept, safe=''))

print('\nPotree cloud.js launcher URL:')
print(launcher + quote(local_cloudjs, safe=''))


## 6) Test checklist
- Open the **EPT** URL in your browser. You should see the cloud streaming in.
- If nothing loads, open dev tools → **Network** tab. Look for **CORS** errors; if present, ensure your server is the CORS version above.
- Very large clouds: consider **retiling** with a larger `--threads` and verify disk throughput.
- If the browser struggles: *downsample first* or split by region/time and build separate EPTs.

## 7) Bonus: Build EPT from multiple files
```bash
entwine build -i /data/*.laz -o /data/out_ept --threads 16
```
## 8) Bonus: Metadata & reprojection
Set or convert spatial reference with **PDAL** to ensure your EPT has a CRS. Example (reproject to EPSG:4978):
```bash
pdal translate in.laz out.laz reprojection --filters.reprojection.out_srs=EPSG:4978
```